# **Data Pre-Processing**


**Covering:**

* EDA Analysis
* Data verification
* Data Quality & Cleaning
* Feature Engineering (Profit/Margin)
* Inventory-linked risk analysis
* validation check against the known stockout/slow-mover labels

 ## **Day 01 - EDA Analysis & Data Verification**

**Loading Libraries**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import os

import warnings
warnings.filterwarnings("ignore")


**Loading Dataset**


**1. Loading Dataset By KaggleHub**

In [ ]:
# from collections import defaultdict
# import kagglehub



# path = kagglehub.dataset_download("mrayyanshehzad/synthetic-retail-dataset-10-million-transactions")

# print("Path to dataset files:", path)

# files = os.listdir(path)
# print("\nParent Folder contain files:", files)


# # Update File Name to select which File to Load


# if 'retail_contaminated_dataset' in files:
#     path = os.path.join(path, 'retail_contaminated_dataset')
#     files = os.listdir(path)
#     print("Sub-folder contain files:", files)


# csv_files = []

# for f in files:
#   if f.endswith(".csv"):
#      csv_files.append(f)

# print("\nCSV files in the folder:", csv_files)


# df = {}

# for file in csv_files:
#     file_path = os.path.join(path, file)

#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} as dataframes['{df_name}'] (Shape: {df[df_name].shape})")

In [ ]:
# !mount | grep kaggle

**Dataset loading issue via Kaggle Hub**
* **ISSUE:** kagglehub returns stale/outdated data in Google Colab
* **ROOT CAUSE:** Colab mounts popular Kaggle datasets from its own server-side  read-only NFS filestore (/kaggle/input/<dataset>), pinned to an old dataset version (versions/1), instead of fetching fresh data via the Kaggle API.
* **Confirmed via:** `mount | grep kaggle` -> shows "...versions/1 ... type nfs (ro,...)"
* This NFS mount is READ-ONLY and cannot be deleted/overridden (chmod/sudo rm fail).
* Verified the actual Kaggle server data IS updated (correct files/sizes) via `kaggle datasets files ...` CLI — so this is a Colab-side caching limitation, not a Kaggle backend issue or a code bug.
* FIX: Bypass kagglehub + /kaggle/input entirely; download fresh via Kaggle CLI into local (non-mounted) /content/ storage.

`!kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/fresh_dataset --unzip`

**Checking Server Side Data**

In [ ]:
# # Server side data sending check

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

# !kaggle datasets files mrayyanshehzad/synthetic-retail-dataset-10-million-transactions


**2. Loading Dataset By Kaggle CLI**

In [ ]:
# !kaggle datasets download mrayyanshehzad/synthetic-retail-dataset-10-million-transactions -p /content/dataset --unzip

In [ ]:

# base_path = "/content/dataset/retail_contaminated_dataset"  # ya retail_clean_dataset
# files = os.listdir(base_path)
# print("Files:", files)

# csv_files = [f for f in files if f.endswith(".csv")]

# df = {}
# for file in csv_files:
#     file_path = os.path.join(base_path, file)
#     df_name = file.replace('.csv', '')
#     df[df_name] = pd.read_csv(file_path)
#     print(f"Loaded: {file} (Shape: {df[df_name].shape})")

**3. Loading Dataset By Local Memory**

In [ ]:
# import  zipfile

# base_path = r"Datasets\retail_contaminated_dataset.zip"

# with zipfile.ZipFile(base_path, 'r' ) as zip_ref:
#     zip_ref.extractall(r"Datasets\extracted_files")


files = os.listdir(r"Datasets\extracted_files")
print("Files:", files)

csv_files = [f for f in files if f.endswith(".csv")]

df = {}
for file in csv_files:
    file_path = os.path.join(r"Datasets\extracted_files", file)
    df_name = file.replace('.csv', '')
    
    df[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file} (Shape: {df[df_name].shape})")

Files: ['customer_master.csv', 'inventory_snapshot.csv', 'processed_sales_transactions.parquet', 'promotions.csv', 'sales_transactions.csv', 'sku_inventory_flags.csv', 'sku_master.csv', 'store_master.csv']
Loaded: customer_master.csv (Shape: (10000, 7))
Loaded: inventory_snapshot.csv (Shape: (26408, 6))
Loaded: promotions.csv (Shape: (100, 8))


**Dataset Loading Flow**

```
📁 /kaggle/input/synthetic-retail-dataset-10-million-transactions (Main Path)
│
├── 📑 README.md
└── 📁 Synthetic Retail Dataset/  <-- [Line: if 'Synthetic Retail Dataset' in files]
    │                                  Enter inside this folder to access .csv files
    ├── 📄 sales_transactions.csv
    ├── 📄 sku_inventory_flags.csv
    ├── 📄 inventory_snapshot.csv
    ├── 📄 store_master.csv
    ├── 📄 customer_master.csv
    ├── 📄 sku_master.csv
    └── 📄 promotions.csv
    
```

**Dataset Relationship**
```
store_master.store_id      ──┐
sku_master.sku_id           ─┼──▶ sales_transactions.csv
customer_master.cust_id     ─┤     (store_id, sku_id, customer_id, promo_id)
promotions.promo_id         ─┘

store_master.store_id   ──┐
sku_master.sku_id        ─┴──▶ inventory_snapshot.csv

sku_master.sku_id  ──▶ sku_inventory_flags.csv   (GROUND-TRUTH labels — do NOT use as a feature,
                                                    only to validate our own risk logic later)
```

### **Step 1 — EDA Analysis**


**Data View, Shape & Columns**

In [ ]:
for name, table in df.items():
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f" {name}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    display(table.head(3))
    print("shape:", table.shape)
    print("columns:", list(table.columns))
    print()

 **Data Info & Describe**

In [ ]:
for name, table in df.items():
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f" {name}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    display(table.info())
    display(table.describe())
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    
    

**Exploring Datasets**

> As our main file is sales_transaction so we focuses on that.

In [ ]:
df['sales_transactions'].dtypes

In [ ]:
df_sales = df['sales_transactions'].iloc[:300000]

In [ ]:
columns = ['quantity', 'unit_price', 'total_value', 'discount_pct']

In [ ]:
for col in columns:
    # 1. Histogram
    sns.histplot(data=df_sales, x=col,kde=True)
    plt.title(f"Histogram for \"{col}\"")
    plt.show()  

    # 2. Boxplot
    sns.boxplot(data=df_sales, y=col)
    # plt.yscale("log") 
    plt.title(f"BoxPlot Distribution of \"{col}\"")
    plt.show() 

    print(
        "-------------------------------------------------------------------------------------------"
    )

### **Step 2 — Verification (before manipulating data)**


**Primary Data Verification**

In [ ]:
expected_rows = {
    "store_master": 30,
    "sku_master": 5000,
    'customer_master': 10000,
    'promotions': 100,
}

for name, expected in expected_rows.items():
    actual = len(df[name])
    # status = "OK" if actual == expected else "MISMATCH"
    status = "Unknown"
    if actual == expected:
      status = "OK"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}   [{status}]")
    elif actual > expected:
      status = "EXCESS"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")
    else:
      status = "MISMATCH"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")

**Secondary Data Verification**

In [ ]:
print(f"{'inventory_snapshot':20s}   actual={len(df['inventory_snapshot']):>8}   expected= no fixed target (quick-check only)")
print(f"{'sales_transactions':20s}   actual={len(df['sales_transactions']):>8}   expected= 10 Million")
print(f"{'sku_inventory_flags':20s}   actual={len(df['sku_inventory_flags']):>8}   expected= 600 (200 STOCKOUT_RISK + 400 SLOW_MOVER)")


In [ ]:
print(df['sku_inventory_flags']['flag'].value_counts())

**Master tables `primary key` uniqueness check**

In [ ]:
primary_key = {
    "store_master": "store_id",
    "sku_master": "sku_id",
    'customer_master': "cust_id",
    'promotions': "promo_id",
}

for name, key in primary_key.items():
    duplicates = df[name][key].duplicated().sum()
    print(f"{name:20s}   duplicates={duplicates:>3}")



**Referential integrity**
 > every FK in sales/inventory must exist in its master table

In [ ]:
checks = [
    #(child table) (child key) (parent table) (parent key)
    ("sales_transactions", "store_id", "store_master", "store_id"),
    ("sales_transactions", "sku_id", "sku_master", "sku_id"),
    ("sales_transactions", "customer_id", "customer_master", "cust_id"),
    ("inventory_snapshot", "store_id", "store_master", "store_id"),
    ("inventory_snapshot", "sku_id", "sku_master", "sku_id"),
]

for child_table, child_key, parent_table, parent_key in checks:

  child_values = df[child_table][child_key]
  parent_values = df[parent_table][parent_key]

  valid = child_values.isin(parent_values).all()

  # print(f"{child_table}.{child_key:13s} -> \t {parent_table}.{parent_key}: {'OK' if valid else 'MISMATCH':}")

  child_full = f"{child_table}.{child_key}"
  parent_full = f"{parent_table}.{parent_key}"
  status = 'OK' if valid else 'MISMATCH'

  print(f"{child_full:32s} ->    {parent_full:26s} :    {status}")


`promo_id` is allowed to be null (no promotion applied) — check only the non-null ones

In [ ]:
promo_values = df['sales_transactions']['promo_id'].dropna()

valid = promo_values.isin(df['promotions']['promo_id']).all()
print(f"sales_transactions.promo_id (non-null) -> promotions.promo_id: {'OK' if valid else 'MISMATCH'}")


## **Day 02 -  Data Quality & Cleaning**

**Also missing before:** an explicit null audit, duplicate check, value-range sanity checks, and a math-consistency check on `total_value`. This is the actual D1/D2 deliverable work — not just `.info()`.

**Get Null Count**

In [ ]:
for name, table in df.items():
  nulls = table.isnull().sum()
  nulls = nulls[nulls > 0]


  print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
  print(f" {name}")
  print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")

  if len(nulls):
    print(nulls)
  else:
    print("no nulls")

  # print(nulls if len(nulls) else "no nulls")


**Insight:**
* `promo_id` is null on most `sales_transactions` rows (as no promo applied - so it's not an error)
* `window_start`/`window_end` is null on all `SLOW_MOVER` rows in `sku_inventory_flags` (as those columns only apply to `STOCKOUT_RISK`).

So I'll Fill/flag them explicitly rather than leaving raw NaN:

In [ ]:
df['sales_transactions'].head()

In [ ]:
# df['sales_transactions']['promo_id'].fillna("NO_PROMO", inplace=True)
df['sales_transactions']['promo_id'].fillna("NO_PROMO")

df['sales_transactions']['promo_applied'] = df['sales_transactions']['promo_id'] != "NO_PROMO"

In [ ]:
df['sku_inventory_flags']['window_start'].isna().sum()

In [ ]:

# df['sku_inventory_flags']['window_start'].fillna(0, inplace=True)
# df['sku_inventory_flags']['window_end'].fillna(0, inplace=True)

**Duplicate rows**

In [ ]:
df['sales_transactions'][df['sales_transactions'].duplicated()]

In [ ]:
duplicates_rows = df['sales_transactions'].duplicated().sum()
print("Duplicated rows in sales_transaction:",duplicates_rows)

In [ ]:
duplicate_id = df['sales_transactions'].duplicated(subset=['receipt_id', 'sku_id']).sum()
print("Duplicate (receipt_id, sku_id) combos:", duplicate_id)

**Insight:** Can be valid but same item added twice on one receipt

**Value-range  checks**

> Avoid negative values

In [ ]:
quantity = (df['sales_transactions']['quantity'] <= 0).sum()
print("quantity <= 0:", quantity)

unit_price = (df['sales_transactions']['unit_price'] <= 0).sum()
print("unit_price <= 0:", unit_price)

discount_pct = (~df['sales_transactions']['discount_pct'].between(0, 100)).sum()
print("discount_pct out of [0,100]:", discount_pct)



stock_on_hand = (df['inventory_snapshot']['stock_on_hand'] < 0).sum()
print("stock_on_hand < 0:", stock_on_hand)

reorder_point = (df['inventory_snapshot']['reorder_point'] < 0).sum()
print("reorder_point < 0:", (df['inventory_snapshot']['reorder_point'] < 0).sum())


age = (~df['customer_master']['age'].between(10, 100)).sum()
print("age out of [10,100]:", age)

**Insight:** There exist no out of bound value.

**Why i choose only these columns to check value-range?**

Reason being selecting only these specific columns is that these are the tables which contains numeric values in whole dataset , rest are either strings or objects

In [ ]:
for name, table in df.items():
  print("-----------------------------------------------------------------------------------------------------------------")
  print(name)
  print("-----------------------------------------------------------------------------------------------------------------")
  table.info()

**Math Consistency Check**

> Does `total_value` logically correct with `quantity` × `price` × `discount`?

In [ ]:
# Copy a sample data
sample = df['sales_transactions'].sample(2000, random_state=42).copy()

# Expected values
sample['expected_total'] = sample['quantity'] * sample['unit_price'] * (1 - sample['discount_pct'] / 100)

# Difference between actual & expected
sample['diff'] = (sample['expected_total'] - sample['total_value']).abs()

print("Rows with > Rupees:1 mismatch:", (sample['diff'] > 1).sum(), "out of", len(sample))

sample[sample['diff'] > 1].head()


**Insight:** our data is mathematically correct.

## **Day 03 - Feature Engineering**

**Calendar features**

Extract Days, Months , Quarter & Years as separate features to better visualize data later on.

>Fixed to sort in real calendar order.

In [ ]:
# df['sales_transactions']['months'] =df['sales_transactions']['date'].apply(extract)
# df['sales_transactions'].head()

In [ ]:
# def extract(txt):
#   data = txt.split("-")[0]
#   data = int(data)
#   return data

In [ ]:
# df['sales_transactions']['years'] = df['sales_transactions']['date'].apply(extract)

In [ ]:
df['sales_transactions']['date'].head()

In [ ]:
df['sales_transactions']['date'] = pd.to_datetime(df['sales_transactions']['date'])


In [ ]:
sales = df['sales_transactions']

In [ ]:
sales['year'] = sales['date'].dt.year

In [ ]:
sales['month_num'] = sales['date'].dt.month
sales

In [ ]:
# def extract(txt):
#   months = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
#   month_num = txt
#   if month_num in months:
#     data = months[month_num]
#   return data

In [ ]:
# sales['month'] = sales['month_num'].apply(extract)

Do the same as above 2 cell code, `strftime` : string format time, `%b` : return short names

In [ ]:
sales['month'] = sales['date'].dt.strftime('%b')

# Defining the order of months
sales['month'] = pd.Categorical(sales['month'], categories=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], ordered=True)

In [ ]:
sales['quarter'] = sales['date'].dt.quarter

In [ ]:
sales['day'] = sales['date'].dt.day_name()

**AI Guide:** Pandas mein `.dt.dayofweek` har date ke liye hafte ka din nikalta hai, lekin yeh humein naam (Monday, Tuesday) nahi deta, balkay 0 se 6 tak ek number deta hai.

> Hafte ke aakhri do din (Saturday aur Sunday) ke numbers 5 aur 6 hain

In [ ]:
sales['is_weekend'] = sales['date'].dt.dayofweek >= 5

In [ ]:

sales.sort_values(by='date', inplace=True)
sales.head()

**Margin**

This was completely missing before. `sku_master.cost_price` was loaded and never used, so no rupee-impact analysis was possible.

In [ ]:
df['sku_master'].head()

In [ ]:
sku_margin = df['sku_master'][['sku_id', 'unit_price', 'cost_price']].copy()

In [ ]:
sku_margin['unit_margin'] = sku_margin['unit_price'] - sku_margin['cost_price']

In [ ]:
sku_margin['margin_pct'] = (sku_margin['unit_margin'] / sku_margin['unit_price'] * 100).round(1)

In [ ]:
sku_margin.head()

In [ ]:
sales = sales.merge(sku_margin[['sku_id', 'cost_price', 'unit_margin', 'margin_pct']], on='sku_id', how='left')

In [ ]:
sales.head()

**Profit**

In [ ]:
sales['total_cost'] = sales['quantity'] * sales['cost_price']

In [ ]:
sales['total_profit'] = sales['total_value'] - sales['total_cost']

In [ ]:
sales['profit_pct'] = (sales['total_profit'] / sales['total_value'] * 100).round(2)

In [ ]:
sales[['sku_id', 'quantity', 'total_value', 'total_cost', 'total_profit', 'profit_pct']].head()

**Redefine the `sales_transaction` dataset**

In [ ]:
df['sales_transactions'] = sales

In [ ]:
df['sales_transactions'].head()

**Products (aka SKUs):**

Become easy for analysis, instead of going merging indirectly to the parent dataset

In [ ]:
df['sku_master'].head()

In [ ]:
products_data = df['sku_master'][['sku_id','sku_name','category',	'subcategory']].copy()

In [ ]:
products_data

In [ ]:
products = df['sales_transactions'].merge(products_data, on='sku_id', how='left')

In [ ]:
products.head()

In [ ]:
products.isna().sum()

In [ ]:
df['sales_transactions'] = products

In [ ]:
df['sales_transactions']

**Customer**

Same it becomes convienet to analyze data of sales with respect to customer  

In [ ]:
df['customer_master']

In [ ]:
customer_data = df['customer_master'][['cust_id',	'age'	,'gender'	,'city']].rename(columns={'cust_id': 'customer_id'})

In [ ]:
customer_data.head()

In [ ]:
customer = df['sales_transactions'].merge(customer_data, on='customer_id', how='left')

In [ ]:
customer.head()

In [ ]:
df['sales_transactions'] = customer

In [ ]:
df['sales_transactions'].head()

## **Checkpointing/Staging Data**

Saving `sales_transaction` as new: file to reduce processor load.

**Mount G-Drive:** In order to store files in google drive for later use.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

**1. Standard CSV Format**

In [ ]:
# df['sales_transactions'].to_csv("Datasets/processed_sales_transaction", index=False)

**2. Parquet Format (The Smart & Fast Way)**

> Because it support Compression & Decompression

In [ ]:
# df['sales_transactions'].to_parquet('/content/drive/MyDrive/Zidio/processed_sales_transactions.parquet')

df['sales_transactions'].to_parquet('Datasets/extracted_files/processed_sales_transactions.parquet')